In [ ]:
import pandas as pd
import baseline_simulator
from utils import *

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

1. Print out objectives for all locations on the grid.
2. Try to change the DCM
3. Fix regular price, only optimize scheduled
4. Change utility function to only care about the minimum
5. Should we multiply by max power rate to convert z_sch to hourly price?

In [ ]:
tou = np.ones((96,)) * 18.6  # off-peak cents/kWh
tou[64:84] = 39.9  # 4 pm - 9 pm peak
tou[36:56] = 16.3  # 9 am - 2 pm super off-peak
TOU = np.concatenate([tou, tou, tou])  # wrap around for multi-day sessions

In [ ]:
sessions_df = pd.read_csv("/Users/sam/Desktop/StationLevelPowerForecasting/data/Sessions3.csv")
sessions_df = sessions_df.sort_values(by='startChargeTime')

test_df = sessions_df[(pd.to_datetime(sessions_df['connectTime']).dt.year == 2024) & (pd.to_datetime(sessions_df['connectTime']).dt.month == 1)]
test_df = test_df[test_df['DurationHrs'] > 0.5]
test_df = test_df[test_df['cumEnergy_Wh'] > 0]
test_df['choice'] = 'SCHEDULED'
test_df = test_df.head(50)
sim = baseline_simulator.BaselineSimulator(test_df, verbose=False, flexibility_constant=0.5)

In [ ]:
power_profiles, prices, hourly_prices = sim.simulate()

In [ ]:
DELTA_T = 0.25

agg_power_profile_all_sch = aggregate_power_profiles(test_df, power_profiles, DELTA_T)
profit_all_sch = get_profit(test_df, power_profiles, prices, DELTA_T, TOU)
profit_all_sch - COST_DC * max(agg_power_profile_all_sch), max(agg_power_profile_all_sch)

In [ ]:
values_index_1

In [ ]:
import matplotlib.pyplot as plt

values_index_0 = [value[0] for value in hourly_prices.values()]
values_index_1 = [value[1] for value in hourly_prices.values()]

plt.figure(figsize=(10, 6))

plt.hist(values_index_0, bins=15, alpha=0.7, color='blue', label="Scheduled")
plt.hist(values_index_1, bins=[263, 265], alpha=0.7, color='red', label="Regular")

plt.xlabel("Hourly Prices")
plt.ylabel("Frequency")
plt.title("Price Distribution")
plt.legend()

plt.grid(axis='y', alpha=0.75)
plt.show()


In [ ]:
from scipy.special import softmax

power_rate = 6.6

# Default discrete choice model parameters
dcm_charging_sch_params = np.array(
[[-power_rate * 0.0184 / 2], [power_rate * 0.0184 / 2], [0], [0]]
)
dcm_charging_reg_params = np.array(
[[power_rate * 0.0184 / 2], [-power_rate * 0.0184 / 2], [0], [0.341]]
)
dcm_leaving_params = np.array(
[[power_rate * 0.005 / 2], [power_rate * 0.005 / 2], [0], [-1]]
)
theta = np.vstack((dcm_charging_sch_params.T, dcm_charging_reg_params.T, dcm_leaving_params.T))


zk = [60, 70, 1, 1]
vk = softmax(theta @ zk).flatten()#.reshape(3,1)
vk

In [ ]:
zk = [1000, 1000, 1, 1]
vk = softmax(theta @ zk).flatten()#.reshape(3,1)
vk

In [ ]:
theta, zk

In [ ]:
from scipy.special import softmax

prices = np.arange(20, 50, 5)
tariff_grid = [(z_sch, z_reg) for z_reg in prices for z_sch in prices if z_reg > z_sch]


for z_sch, z_reg in tariff_grid:
    zk = [z_sch, z_reg, 1, 1]
    vk = softmax(theta @ zk).flatten()#.reshape(3,1)
    print(vk[1])